# Act 2 · Atlas — Mapping Models to HRA Anatomy

_Investigation `hra-integration` — coder reproduction notebook._

**Question.** Can we pull HRA datasets/knowledge via the live CCF API (reference organs,
cell-type terms, anatomical-structure terms) and link BioModels to HRA
anatomy (Uberon organs) via a transparent biomodel Digital Object +
organ->models index — and, having built that index, actually get the
resulting models onto the HRA atlas as something a user can see?

**Act 2 of 4** in the viva-human-atlas arc — Trust → **Atlas** → Coverage → Connectivity.
Takes the trustworthy models from Act 1 and gives each one an anatomical
address. Builds the HRA (Human Reference Atlas) side of the
fetch-and-compare spine, then closes the loop into the Atlas Browser: a
thin CCF-API
client (reference organs, cell-type terms, anatomical-structure terms);
a biomodel Digital Object annotation layer with two complementary organ
matchers — name-synonym matching over BioModels titles, and
annotation-based matching over each model's own SBML MIRIAM/BTO
metadata; a measured comparison of how much recall the annotation
matcher adds over name-only matching; and the HRA Computational Model Atlas
(`studies/hra-atlas-browser/viz/atlas/`) that renders the resulting
organ->models index as an interactive 3D atlas a user can actually see
biomodels attached to.
← Prev: **Act 1 · Model Trust** (`glucose-regulation`) · → Next: **Act 3 · Coverage** (`hra-3d`).

---

**How to use this notebook**

1. **Run ▸ Run All Cells** to reproduce every study end to end.
2. To explore, edit a study's **Parameters** cell, then re-run from its **Run** section — the figures regenerate with your values.
3. The **Setup** cell at the top is collapsed scaffolding (environment + render helpers); expand it only to inspect. Set `RERUN = False` there to render the committed `runs.db` without re-simulating.

The prose states the **question and parameters**; the figures each run produces are the results.


In [ ]:
"""Setup — environment & helpers (runs automatically; open only to inspect).

Generated by vivarium-workbench (notebook_export). This cell resolves the repo,
registers the workspace Core, and defines the render helpers; the science lives
in the sections below. Each study is re-run live with the workspace's own
process-bigraph protocol and its figures rendered from the resulting runs.db.
"""
import os
import sys
from pathlib import Path

# Resolve the repository root robustly so this notebook runs from a fresh clone
# at ANY path with no setup (no env var, no path editing). Priority:
#   1. $VIVARIUM_REPO, if it points at a real directory;
#   2. walk up from the notebook's working directory for the repo markers
#      (a directory holding both 'workspace/' and 'pyproject.toml') — Jupyter
#      starts in the notebook's dir, so a committed notebook finds its own root;
#   3. the absolute path it was generated for (back-compat for old layouts);
#   4. the current working directory (last resort).
def _find_repo_root(_start):
    for _cand in (_start, *_start.parents):
        if (_cand / "workspace").is_dir() and (_cand / "pyproject.toml").is_file():
            return _cand
    return None

REPO = None
_env = os.environ.get("VIVARIUM_REPO")
if _env and Path(_env).is_dir():
    REPO = Path(_env)
if REPO is None:
    REPO = _find_repo_root(Path.cwd().resolve())
if REPO is None and Path('/Users/eranagmon/code/viva-human-atlas--model-connectivity').is_dir():
    REPO = Path('/Users/eranagmon/code/viva-human-atlas--model-connectivity')
if REPO is None:
    REPO = Path.cwd()
sys.path.insert(0, str(REPO))
# Composite specs use repo-root-relative paths (datasets, caches), and the
# workspace's runner/renderer assume cwd == repo root — so run from there.
os.chdir(REPO)

# Re-simulate from scratch? Set False to render the committed runs.db (fast).
RERUN = True

# Keep the output clean: composite discovery and the pydantic models emit
# benign UserWarnings (optional packages it skips, a field-name shadow) and the
# simulators emit DeprecationWarnings — none are actionable from a reproduction
# notebook, so silence those two categories. Remove this to see them.
import warnings as _warnings
_warnings.filterwarnings("ignore", category=UserWarning)
_warnings.filterwarnings("ignore", category=DeprecationWarning)

# --- standard process-bigraph protocol: register the workspace's Core ---
from viva_human_atlas.core import build_core
core = build_core()

# --- imported from the repo this notebook was generated for ---

from IPython.display import HTML, display

import contextlib as _contextlib, io as _io
@_contextlib.contextmanager
def quiet():
    """Silence the simulator's verbose per-step stdout so the notebook
    output stays readable (the figures below are the results)."""
    with _contextlib.redirect_stdout(_io.StringIO()):
        yield

import html as _htmlmod
def show_viz(_h, height=560):
    """Display a visualization's HTML in an isolated iframe.

    The figures embed their own scripts (e.g. Plotly); JupyterLab does not
    execute <script> tags from display(HTML(...)), so an iframe srcdoc is
    used instead — the browser runs the scripts inside the frame."""
    display(HTML(
        '<iframe srcdoc="{}" style="width:100%;height:{}px;border:0">'
        '</iframe>'.format(_htmlmod.escape(_h, quote=True), height)
    ))

import json as _json
def describe_spec(spec):
    """Print a composite spec's structure (parameters, processes, wiring)
    then the full editable dict. The spec is plain data — assign to any
    field (e.g. spec['state'][proc]['config'][...]) before building."""
    print("composite:", spec.get("name"))
    if spec.get("description"):
        print("description:", str(spec["description"]).strip())
    _params = spec.get("parameters") or {}
    if _params:
        print("\nparameters (filled into ${name} placeholders):")
        for _p, _pdef in _params.items():
            print(f"  {_p}: default={_pdef.get('default')!r}  type={_pdef.get('type')}")
    print("\nprocesses (node -> address):")
    for _node, _body in (spec.get("state") or {}).items():
        if not (isinstance(_body, dict) and _body.get("_type") == "process"):
            continue
        print(f"  {_node}  ->  {_body.get('address')}   interval={_body.get('interval')!r}")
        for _port in ("inputs", "outputs"):
            if _body.get(_port):
                print(f"      {_port} ports: {_body[_port]}")
    print("\nfull editable spec dict:")
    print(_json.dumps(spec, indent=2, default=str))

def materialize_composite(spec_id, overrides=None):
    """Build a composite into an editable payload {name, description,
    parameters, state, _overrides, _id}. Works for @composite_generator
    functions via the process-bigraph registry (the common case); ``state``
    is the built document (process nodes at top level)."""
    import importlib
    from process_bigraph.composite_generator import (
        _REGISTRY, apply_core_extensions, build_generator)
    try:
        from process_bigraph.composite_generator import discover_generators
        discover_generators()
    except Exception:
        pass
    entry = _REGISTRY.get(spec_id)
    if entry is None:
        # editable installs are invisible to packages_distributions, so
        # discover alone can miss them — prime the module the id names.
        try:
            importlib.import_module(spec_id.rsplit('.', 1)[0])
            from process_bigraph.composite_generator import discover_generators as _dg
            _dg()
        except Exception:
            pass
        entry = _REGISTRY.get(spec_id)
    if entry is None:
        raise KeyError('composite not registered: ' + repr(spec_id))
    params = dict(entry.parameters or {})
    ov = {k: (v.get('default') if isinstance(v, dict) else None)
          for k, v in params.items()}
    if overrides:
        ov.update({k: val for k, val in overrides.items() if k in params})
    _c = apply_core_extensions(entry, core)
    try:
        doc = build_generator(entry, ov or None, core=_c)
    except TypeError:
        doc = entry.func(_c, **(ov or {}))
    return {'name': entry.name, 'description': entry.description,
            'parameters': params, 'state': doc, '_overrides': ov, '_id': spec_id}

import base64 as _b64, pathlib as _pl
_STATIC_VIZ_SCHEMES = ('image:', 'file:', 'gif:', 'png:', 'svg:', 'jpg:', 'jpeg:')
def _render_viz_process(address, config):
    """Render a Visualization process/step (e.g. ``local:<Class>``) to
    HTML: wrap it as a one-step composite and run it once — the viz writes
    Plotly HTML to its ``html`` output port. Mirrors the workbench's own
    env-worker viz path. Self-running vizes need no run data."""
    from process_bigraph import Composite
    _doc = {
        'inputs_store': {}, 'output_store': '',
        'visualization': {
            '_type': 'step', 'address': address,
            'config': {k: v for k, v in (config or {}).items()
                       if k not in ('inputs_map', 'sources')},
            'inputs': {}, 'outputs': {'html': ['output_store']},
        },
    }
    _comp = Composite({'state': _doc}, core=core)
    _comp.run(1)
    _html = _comp.state.get('output_store')
    if isinstance(_html, dict):
        _html = _html.get('value') or _html.get('_value') or ''
    return _html if isinstance(_html, str) else ''
def _render_one(address, config, runs_db, study_yaml, overrides=None):
    """Generic figure renderer (no workspace render_study_viz.py).
    A static ``image:``/``file:`` (or bare-path) address embeds the file;
    any other address (``local:<VizProcess>``) is run as a Visualization
    and its HTML returned. ``overrides`` (the composite's edited parameter
    dict) is merged into the viz config first so a self-running figure
    reflects edited parameters; a viz that does not accept those keys
    falls back to the base config."""
    addr = str(address or '')
    _head = addr.split('/', 1)[0]
    _is_static = (not addr) or (':' not in _head) or addr.startswith(_STATIC_VIZ_SCHEMES)
    if not _is_static:
        if core is None:
            return f'<p style="color:#b91c1c">no core to render {address}</p>'
        _base = {k: v for k, v in (config or {}).items()}
        _tries = ([{**_base, **overrides}] if overrides else []) + [_base]
        _err = ''
        for _cfg in _tries:
            try:
                _h = _render_viz_process(addr, _cfg)
            except Exception as _e:
                _err = (f'<p style="color:#b91c1c">visualization {address} not renderable yet: '
                        f'{type(_e).__name__}: {_e}</p>')
                continue
            return _h or f'<p style="color:#6b7280">{address} produced no HTML</p>'
        return _err
    for _scheme in _STATIC_VIZ_SCHEMES:
        if addr.startswith(_scheme):
            addr = addr[len(_scheme):]; break
    _p = _pl.Path(addr)
    if not _p.is_absolute():
        _p = _pl.Path(study_yaml).resolve().parent / _p
    if not _p.is_file():
        return f'<p style="color:#b91c1c">figure not found: {address}</p>'
    _suffix = _p.suffix.lower()
    if _suffix == '.svg':
        return _p.read_text(encoding='utf-8', errors='replace')
    if _suffix in ('.png', '.jpg', '.jpeg', '.gif', '.webp'):
        _mime = 'jpeg' if _suffix in ('.jpg', '.jpeg') else _suffix[1:]
        _data = _b64.b64encode(_p.read_bytes()).decode('ascii')
        return f'<img src="data:image/{_mime};base64,{_data}" style="max-width:100%"/>'
    if _suffix in ('.html', '.htm'):
        return _p.read_text(encoding='utf-8', errors='replace')
    return f'<p style="color:#6b7280">unsupported figure type: {address}</p>'

## Study: HRA Reference Organs — Uberon-Keyed Organ Set (`hra-reference-organs`)

**Question.** Does the HRA CCF API's reference-organs endpoint load into a Step-driven
composite, giving an Uberon-keyed organ set (with per-sex GLB assets)?


### Parameters


### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: hra-reference-organs ===
STUDY = 'hra-reference-organs'
STUDY_DIR = REPO / 'studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No baseline composite declared for this study — nothing to build.")

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| reference-organs-nonempty _(primary)_ |  |  |
| uberon-keyed-majority _(primary)_ |  |  |
| per-sex-split _(supporting)_ |  |  |


### Literature anchors

| expectation | model observable | source | status |
| --- | --- | --- | --- |
| The HRA provides a per-sex, Uberon-keyed whole-organ reference set as the spatial scaffold of the atlas. | 81 reference-organ entries loaded; 69 UBERON-keyed; split 45 female-specific / 36 male-specific. | Boerner et al. 2021, the Human Reference Atlas / Common Coordinate Framework. | Verified |
| Whole organs are identified by Uberon anatomy CURIEs. | 69/81 entries resolve to a UBERON: CURIE (the remainder to FMA). | Mungall et al. 2012, Uberon integrated anatomy ontology. | Verified |


## Study: HRA Cell-Type Term Occurrences (`hra-cell-types`)

**Question.** Does the HRA CCF API's cell-type-term-occurences endpoint load into a
Step-driven composite, giving a Cell Ontology (CL) term occurrence list?


### Parameters


### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: hra-cell-types ===
STUDY = 'hra-cell-types'
STUDY_DIR = REPO / 'studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No baseline composite declared for this study — nothing to build.")

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| cl-terms-nonempty-sorted _(primary)_ |  |  |
| top-term-dominates _(supporting)_ |  |  |


### Literature anchors

| expectation | model observable | source | status |
| --- | --- | --- | --- |
| HRA annotates cell populations across its datasets with Cell Ontology (CL) terms. | 149 distinct CL terms returned, count-sorted by occurrence. | Diehl et al. 2016, Cell Ontology. | Verified |


## Study: HRA Anatomical-Structure Term Occurrences (`hra-anatomical-structures`)

**Question.** Does the HRA CCF API's ontology-term-occurences endpoint load into a
Step-driven composite, giving an anatomical-structure term occurrence list?


### Parameters


### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: hra-anatomical-structures ===
STUDY = 'hra-anatomical-structures'
STUDY_DIR = REPO / 'studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No baseline composite declared for this study — nothing to build.")

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| anatomical-terms-nonempty-sorted _(primary)_ |  |  |
| cross-endpoint-consistency _(supporting)_ |  |  |


### Literature anchors

| expectation | model observable | source | status |
| --- | --- | --- | --- |
| HRA records which anatomical structures its datasets sample from, keyed to Uberon anatomy terms. | 288 distinct anatomical-structure terms; top term UBERON:0013702 at 2,762 occurrences. | Boerner et al. 2021 (HRA/CCF); Mungall et al. 2012 (Uberon). | Verified |


## Study: Glucose BioModel Digital Objects — Uberon Organ Annotation (`glucose-biomodel-do`)

**Question.** Can glucose-regulation BioModels be annotated with HRA Uberon organ terms
(via transparent synonym matching) and inverted into an organ->models index?


### Parameters


### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: glucose-biomodel-do ===
STUDY = 'glucose-biomodel-do'
STUDY_DIR = REPO / 'studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No baseline composite declared for this study — nothing to build.")

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| at-least-one-organ-tagged _(primary)_ |  |  |
| pancreas-dominates _(primary)_ |  |  |
| organ-index-inspectable _(supporting)_ |  |  |


### Literature anchors

| expectation | model observable | source | status |
| --- | --- | --- | --- |
| A BioModels SBML entry can be given an explicit HRA anatomical address (a Uberon organ). | organ_to_models maps pancreas (UBERON:0001264, 4 models) and liver (UBERON:0002107, 1 model). | Malik-Sheriff et al. 2020 (BioModels); Mungall et al. 2012 (Uberon). | Partial (name-only matching, 5/25 tagged). |


## Study: Annotation-Based Organ Matching — SBML MIRIAM + BTO Crosswalk Provenance (`annotation-organ-matching`)

**Question.** How does a BioModels entry get tagged with an HRA Uberon organ when the
match comes from its embedded SBML MIRIAM annotations rather than its
name — and which biological CVTerm qualifiers / ontologies actually carry
that organ signal?


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `baseline` | `viva_human_atlas.composites.annotation_composite.annotation-organ-matching` | 0 | catalog_path=datasets/biomodel_annotation_catalog.json |


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**The model — `viva_human_atlas.composites.annotation_composite.annotation-organ-matching`**

Summarize the committed annotation-based (SBML MIRIAM + BTO crosswalk) organ-matching catalog: model/organ counts and the biological-qualifier/ontology distribution behind each organ tag.


In [ ]:
# === Parameters — edit any value, then re-run from the Run cell below ===
# Each entry is the composite's current default; the comment explains it.
ov_viva_human_atlas_composites_annotation_composite_annotation_organ_matching = {
    'catalog_path': 'datasets/biomodel_annotation_catalog.json',   # Path to the committed annotation catalog JSON.
}

### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: annotation-organ-matching ===
STUDY = 'annotation-organ-matching'
STUDY_DIR = REPO / 'studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# This study has no *recorded* runs yet. RERUN builds its declared
# composite(s) live and runs them; a composite whose module processes
# are not implemented yet reports the error instead of crashing.
STEPS_baseline = 100
INTERVAL_baseline = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_human_atlas_composites_annotation_composite_annotation_organ_matching if 'ov_viva_human_atlas_composites_annotation_composite_annotation_organ_matching' in dir() else None  # edits from the panel cell, if any
            spec_viva_human_atlas_composites_annotation_composite_annotation_organ_matching = materialize_composite('viva_human_atlas.composites.annotation_composite.annotation-organ-matching', _ov)
            _comp = Composite({'state': spec_viva_human_atlas_composites_annotation_composite_annotation_organ_matching['state']}, core=core)
            _comp.run(STEPS_baseline)
        print('ran', 'baseline', f'(viva_human_atlas.composites.annotation_composite.annotation-organ-matching) for', STEPS_baseline, 'steps')
    except Exception as _e:
        print(f"baseline not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| bto-crosswalk-dominates _(primary)_ |  |  |
| corpus-organ-coverage _(primary)_ |  |  |
| occurs-in-is-qualifiers-dominate _(supporting)_ |  |  |


### Literature anchors

| expectation | model observable | source | status |
| --- | --- | --- | --- |
| BioModels curators annotate SBML elements with MIRIAM CVTerms (biological qualifier + ontology resource URI), far more often with tissue (BTO) than with direct anatomy (Uberon/FMA). | 168 CVTerm hits: ontology BTO 156 vs UBERON 12; qualifiers BQB_OCCURS_IN 88, BQB_IS 40, BQB_IS_VERSION_OF 35. | Juty et al. 2013 (MIRIAM/SBO qualifiers); Malik-Sheriff et al. 2020 (BioModels curation). | Verified |
| Resolved organ terms map to Uberon anatomy. | 14 distinct HRA organs reached across 114 tagged models. | Mungall et al. 2012, Uberon. | Verified |


## Study: Annotation Recall Gain — Annotation-Matching vs Name-Only Organ Tagging (`annotation-recall-gain`)

**Question.** How much organ/model coverage does annotation-based organ matching (SBML
MIRIAM + BTO crosswalk) add over the existing name-synonym matcher
(`glucose-biomodel-do`/`corpus-coverage`), across the full 1,096-model
curated BioModels corpus?


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `baseline` | `viva_human_atlas.composites.annotation_composite.annotation-recall-gain` | 0 | name_catalog_path=datasets/biomodel_corpus_catalog.json, annotation_catalog_path=datasets/biomodel_annotation_catalog.json |


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**The model — `viva_human_atlas.composites.annotation_composite.annotation-recall-gain`**

Compare the annotation-based organ-matching catalog against the name-synonym catalog to quantify the recall gain (organs/models added) from parsing SBML MIRIAM/BTO annotations.


In [ ]:
# === Parameters — edit any value, then re-run from the Run cell below ===
# Each entry is the composite's current default; the comment explains it.
ov_viva_human_atlas_composites_annotation_composite_annotation_recall_gain = {
    'name_catalog_path'      : 'datasets/biomodel_corpus_catalog.json',   # Path to the committed name-synonym catalog JSON.
    'annotation_catalog_path': 'datasets/biomodel_annotation_catalog.json',   # Path to the committed annotation catalog JSON.
}

### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: annotation-recall-gain ===
STUDY = 'annotation-recall-gain'
STUDY_DIR = REPO / 'studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# This study has no *recorded* runs yet. RERUN builds its declared
# composite(s) live and runs them; a composite whose module processes
# are not implemented yet reports the error instead of crashing.
STEPS_baseline = 100
INTERVAL_baseline = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_human_atlas_composites_annotation_composite_annotation_recall_gain if 'ov_viva_human_atlas_composites_annotation_composite_annotation_recall_gain' in dir() else None  # edits from the panel cell, if any
            spec_viva_human_atlas_composites_annotation_composite_annotation_recall_gain = materialize_composite('viva_human_atlas.composites.annotation_composite.annotation-recall-gain', _ov)
            _comp = Composite({'state': spec_viva_human_atlas_composites_annotation_composite_annotation_recall_gain['state']}, core=core)
            _comp.run(STEPS_baseline)
        print('ran', 'baseline', f'(viva_human_atlas.composites.annotation_composite.annotation-recall-gain) for', STEPS_baseline, 'steps')
    except Exception as _e:
        print(f"baseline not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| annotation-adds-recall _(primary)_ |  |  |
| union-still-small-fraction _(primary)_ |  |  |
| matchers-complementary _(supporting)_ |  |  |


### Literature anchors

| expectation | model observable | source | status |
| --- | --- | --- | --- |
| A model's curated SBML MIRIAM metadata encodes anatomical context its title need not spell out, so annotation matching can reach organs a title never names. | Annotation matching reaches 4 organs (lymph-node, ovary, spleen, urinary-bladder) name matching never sees; +85 models overall. | Juty et al. 2013 (MIRIAM); Malik-Sheriff et al. 2020 (BioModels). | Verified |


## Study: BioModels → HRA Map — Harvesting the Curated Corpus into the Human Reference Atlas (`biomodel-hra-map`)

**Question.** Can we harvest the entire curated BioModels corpus into a single, reusable
JSON DB that links each model to the Human Reference Atlas — molecular
identifiers (CHEBI/UniProt/KEGG/GO/Reactome), publication link, organism,
and HRA organ / FTU / cell-type mapping (Uberon/CL) — as a downloadable
workspace resource the Atlas Browser and downstream coverage studies can
key off, and how much of the corpus does each annotation actually reach?


### Parameters


### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: biomodel-hra-map ===
STUDY = 'biomodel-hra-map'
STUDY_DIR = REPO / 'studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No baseline composite declared for this study — nothing to build.")

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| full-corpus-harvested _(primary)_ |  |  |
| molecular-coverage-dense _(primary)_ |  |  |
| hra-anatomy-recall-limited _(primary)_ |  |  |


### Literature anchors

| expectation | model observable | source | status |
| --- | --- | --- | --- |
| The HRA organizes the body as organ -> anatomical structure -> functional tissue unit -> cell type. | Per-model HRA mapping reaches organs 264 (24%), FTUs 163 (15%), cell types 363 (33%), Uberon 323 (29%). | Boerner et al. 2021, the Human Reference Atlas. | Partial (recall-limited anatomy). |
| Functional tissue units (FTUs) are the HRA's functional anatomical subunits of an organ. | 163 models (15%) map to an HRA FTU. | Bidanta et al. 2025, HRA functional tissue units. | Partial |
| BioModels curates dense molecular annotations (CHEBI/UniProt/KEGG/GO) in each model's SBML. | GO 967 (88%), UniProt 507 (46%), CHEBI 480 (44%), KEGG 322 (29%); 978 (89%) carry >=1 molecular id. | Malik-Sheriff et al. 2020, BioModels. | Verified |


## Study: Unified Model Harvest — BioModels + PhysioNet into the HRA Model DB (`model-harvest`)

**Question.** Can a single Step reproduce/refresh the unified HRA model DB
(datasets/model_hra_map.json) across every registered model source --
BioModels and, newly, PhysioNet -- incrementally and non-destructively, so
that running this study *is* the reproducibility hook for the whole DB,
not just a read of a static file?


### Parameters


### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: model-harvest ===
STUDY = 'model-harvest'
STUDY_DIR = REPO / 'studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No baseline composite declared for this study — nothing to build.")

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| per-source-counts-sum _(primary)_ |  |  |
| incremental-non-destructive _(primary)_ |  |  |
| network-free-baseline _(supporting)_ |  |  |


### Literature anchors

| expectation | model observable | source | status |
| --- | --- | --- | --- |
| BioModels is the curated repository of published quantitative systems-biology models. | 1,096 BioModels rows harvested into the unified HRA model DB (repository=biomodels). | Malik-Sheriff et al. 2020, BioModels. | Verified |


## Open decisions
- Resolved: SBML MIRIAM/BTO annotation matching (annotation-organ-matching, annotation-recall-gain) raised full-corpus recall from 82/1,096 (7.5%) to a 167/1,096 (15.2%) union across 14 organs. Should the union (not name-only) become the default organ->models index the Atlas Browser and downstream coverage studies key off?
- How much more organ coverage would extending the 32-entry BTO->organ crosswalk unlock?
